# Home Health DME Analytics — Snowflake SnowDay Hands-On Lab

Welcome to the **SnowDay**! This notebook builds a complete analytics platform for a home health DME (Durable Medical Equipment) company.

| Use Case | Source Data | Goal |
|----------|-------------|------|
| **Denials Reduction** | Claims / EDI 835 | Reduce 17% denial rate to <5%, recover $4M+ revenue |
| **Sales Data Analysis** | CRM + CMS Medicare | Territory optimization, market penetration |
| **Call Center Consolidation** | Avaya, Five9, RingCentral | Unified KPIs across 700+ locations |

### What You'll Build
1. `RAW_DATA` schema — 224,000+ records from Q1 2026 CSVs
2. `TRANSFORMED` schema — **Dynamic Tables** medallion (Bronze → Silver → Gold)
3. `ANALYTICS` schema — Domain KPIs + Semantic View for Cortex Analyst
4. **HIPAA Governance** — Dynamic masking, row access, PHI classification
5. **Cortex Search** — RAG over 10 operational policy documents
6. **Intelligence Agent** — Combines Cortex Analyst + Cortex Search
7. **FHIR R4 Pipeline** — EHR data via VARIANT + LATERAL FLATTEN
8. **Streamlit Dashboard** — 5-tab app with embedded AI assistant

> All data is synthetic. No real PHI. Run cells in order.

## Section 1: Environment Setup

**SNOWFLAKE DIFFERENTIATOR:** Virtual Warehouses auto-suspend after idle time (you pay $0 when not querying). No Fabric F-SKU running 24/7.

| Resource | Purpose | Size |
|----------|---------|------|
| `HOME_HEALTH_LOAD_WH` | ETL loading | MEDIUM, 60s suspend |
| `HOME_HEALTH_ANALYTICS_WH` | Analytics + AI | LARGE, auto-scale 1-5 clusters |
| `HOME_HEALTH_ADHOC_WH` | Ad-hoc exploration | SMALL, 60s suspend |

| Role | Access |
|------|--------|
| `DATA_ENGINEER` | Pipeline + infrastructure |
| `BILLING_ADMIN` | Claims, denials, appeals |
| `SALES_MANAGER` | CRM, referrals, CMS data |
| `CALL_CENTER_LEAD` | CDRs, agent performance |
| `ANALYST` | Cross-functional read |
| `EXECUTIVE` | All KPI views |

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HOME_HEALTH_DEMO
  COMMENT = 'Home Health DME Analytics Demo - Denials, Sales, Call Center';

CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.RAW_DATA      COMMENT = 'Raw data from source systems';
CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.TRANSFORMED   COMMENT = 'Cleaned data via Dynamic Tables';
CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.ANALYTICS     COMMENT = 'KPIs, Semantic View, analytics';
CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.DOCUMENTS     COMMENT = 'Policy documents for Cortex Search';
CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.FHIR_RAW      COMMENT = 'Raw FHIR R4 bundles (VARIANT)';
CREATE OR REPLACE SCHEMA HOME_HEALTH_DEMO.FHIR_ANALYTICS COMMENT = 'Flattened FHIR views';

CREATE OR REPLACE ROLE DATA_ENGINEER    COMMENT = 'Data pipeline and infrastructure';
CREATE OR REPLACE ROLE BILLING_ADMIN    COMMENT = 'Revenue cycle and denials';
CREATE OR REPLACE ROLE SALES_MANAGER    COMMENT = 'Sales and territory management';
CREATE OR REPLACE ROLE CALL_CENTER_LEAD COMMENT = 'Call center operations';
CREATE OR REPLACE ROLE ANALYST          COMMENT = 'Cross-functional analytics';
CREATE OR REPLACE ROLE EXECUTIVE        COMMENT = 'C-suite KPI access';

GRANT ROLE BILLING_ADMIN    TO ROLE EXECUTIVE;
GRANT ROLE SALES_MANAGER    TO ROLE EXECUTIVE;
GRANT ROLE CALL_CENTER_LEAD TO ROLE EXECUTIVE;
GRANT ROLE ANALYST          TO ROLE EXECUTIVE;
GRANT ROLE EXECUTIVE        TO ROLE SYSADMIN;
GRANT ROLE DATA_ENGINEER    TO ROLE SYSADMIN;

CREATE OR REPLACE WAREHOUSE HOME_HEALTH_LOAD_WH
  WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
  COMMENT = 'ETL: auto-suspends 60s. Zero cost when idle.';

CREATE OR REPLACE WAREHOUSE HOME_HEALTH_ANALYTICS_WH
  WAREHOUSE_SIZE = 'LARGE' AUTO_SUSPEND = 300 AUTO_RESUME = TRUE
  MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 5 SCALING_POLICY = 'STANDARD'
  COMMENT = 'Analytics: auto-scales 1-5 clusters for concurrent queries.';

CREATE OR REPLACE WAREHOUSE HOME_HEALTH_ADHOC_WH
  WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE DATABASE HOME_HEALTH_DEMO;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;
SELECT 'Environment ready' AS status, CURRENT_DATABASE() AS db, CURRENT_WAREHOUSE() AS wh;

## Section 2: Stages and Data Upload

All CSV files are pre-generated in the `data/` folder. Upload to Snowflake stages before loading.

**Using SnowSQL (run from the cloned repo directory):**
```sql
USE DATABASE HOME_HEALTH_DEMO; USE SCHEMA RAW_DATA;
PUT file://./data/locations.csv              @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/payer_contracts.csv        @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/claims_submissions.csv     @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/claims_denials.csv         @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/denial_appeals.csv         @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/sales_rep_activity.csv     @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/physician_referrals.csv    @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/cms_respiratory_claims.csv @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/call_detail_records.csv    @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/call_agent_performance.csv @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/patient_satisfaction.csv   @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/home_health_fhir_bundle.json @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
```

Also upload all 10 policy docs from `documents/` to `HOME_HEALTH_DOCUMENTS_STAGE`.

> Use **Snowsight UI**: Data > Databases > HOME_HEALTH_DEMO > RAW_DATA > Stages > Upload button.

In [ ]:
USE ROLE DATA_ENGINEER;
USE DATABASE HOME_HEALTH_DEMO;
USE SCHEMA RAW_DATA;
USE WAREHOUSE HOME_HEALTH_LOAD_WH;

CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1 NULL_IF = ('', 'NULL', 'None') EMPTY_FIELD_AS_NULL = TRUE;

CREATE OR REPLACE FILE FORMAT JSON_FORMAT
    TYPE = 'JSON' STRIP_OUTER_ARRAY = FALSE;

CREATE OR REPLACE STAGE HOME_HEALTH_DATA_STAGE
    FILE_FORMAT = CSV_FORMAT COMMENT = 'All Home Health demo data files';

CREATE OR REPLACE STAGE HOME_HEALTH_DOCUMENTS_STAGE
    COMMENT = 'Policy documents for Cortex Search';

LIST @HOME_HEALTH_DATA_STAGE;
SELECT 'Stages created' AS status;

## Section 3: Load Data (224K+ Records)

| Table | Records | Description |
|-------|---------|-------------|
| LOCATIONS | 700 | Operating centers, 48 states |
| CLAIMS_SUBMISSIONS | 50,000 | All Q1 2026 DME claims |
| CLAIMS_DENIALS | 8,459 | Denied claims with CARC/RARC codes |
| DENIAL_APPEALS | 4,643 | Appeal outcomes + recovery |
| SALES_REP_ACTIVITY | 15,000 | CRM activities by 150 reps |
| PHYSICIAN_REFERRALS | 12,000 | Physician-to-patient referrals |
| CMS_RESPIRATORY_CLAIMS | 30,000 | CMS Medicare Q4 2025 market data |
| CALL_DETAIL_RECORDS | 100,000 | CDRs from Avaya, Five9, RingCentral |
| CALL_AGENT_PERFORMANCE | 500 | Agent benchmarks |
| PATIENT_SATISFACTION | 8,000 | Post-call surveys |

> `COPY INTO` with `FORCE = TRUE` avoids load history conflicts — useful when re-running the HOL.

In [ ]:
CREATE OR REPLACE TABLE LOCATIONS (
    location_id VARCHAR(10), location_name VARCHAR(200), address VARCHAR(200),
    city VARCHAR(100), state VARCHAR(2), zip_code VARCHAR(10), region VARCHAR(50),
    phone VARCHAR(20), manager VARCHAR(100), open_date DATE, is_active BOOLEAN,
    square_footage INT, employee_count INT);

CREATE OR REPLACE TABLE PAYER_CONTRACTS (
    contract_id VARCHAR(10), payer_code VARCHAR(20), payer_name VARCHAR(100),
    plan_type VARCHAR(50), effective_date DATE, termination_date DATE,
    reimbursement_rate_pct FLOAT, timely_filing_days INT,
    prior_auth_required BOOLEAN, cmn_required BOOLEAN,
    electronic_submission BOOLEAN, contact_phone VARCHAR(20));

CREATE OR REPLACE TABLE CLAIMS_SUBMISSIONS (
    claim_id VARCHAR(15), patient_id VARCHAR(15), location_id VARCHAR(10),
    payer_code VARCHAR(20), payer_name VARCHAR(100), hcpcs_code VARCHAR(10),
    equipment_name VARCHAR(100), equipment_category VARCHAR(50), quantity INT,
    billed_amount FLOAT, allowed_amount FLOAT, submission_date DATE,
    service_date DATE, referring_physician_npi VARCHAR(15),
    referring_physician_name VARCHAR(100), physician_specialty VARCHAR(50),
    diagnosis_code VARCHAR(10), modifier VARCHAR(5), place_of_service VARCHAR(5),
    claim_status VARCHAR(20), adjudication_date DATE, paid_amount FLOAT,
    has_cmn BOOLEAN, prior_auth_obtained BOOLEAN);

CREATE OR REPLACE TABLE CLAIMS_DENIALS (
    denial_id VARCHAR(12), claim_id VARCHAR(15), patient_id VARCHAR(15),
    location_id VARCHAR(10), payer_code VARCHAR(20), payer_name VARCHAR(100),
    hcpcs_code VARCHAR(10), equipment_category VARCHAR(50), denial_date DATE,
    denial_code VARCHAR(10), denial_reason VARCHAR(200), denial_category VARCHAR(50),
    root_cause VARCHAR(50), billed_amount FLOAT, denied_amount FLOAT,
    assigned_to VARCHAR(100), priority VARCHAR(10), status VARCHAR(20),
    days_to_resolve INT, is_repeat_denial BOOLEAN, original_claim_clean BOOLEAN);

CREATE OR REPLACE TABLE DENIAL_APPEALS (
    appeal_id VARCHAR(12), denial_id VARCHAR(12), claim_id VARCHAR(15),
    location_id VARCHAR(10), payer_code VARCHAR(20), appeal_date DATE,
    appeal_level VARCHAR(30), appeal_reason VARCHAR(200), supporting_docs VARCHAR(100),
    outcome VARCHAR(20), outcome_date DATE, recovered_amount FLOAT,
    appeal_specialist VARCHAR(100), turnaround_days INT, payer_response_notes VARCHAR(500));

CREATE OR REPLACE TABLE SALES_REP_ACTIVITY (
    activity_id VARCHAR(12), rep_id VARCHAR(10), rep_name VARCHAR(100),
    territory VARCHAR(50), activity_date DATE, activity_type VARCHAR(30),
    physician_npi VARCHAR(15), physician_name VARCHAR(100), physician_specialty VARCHAR(50),
    facility_name VARCHAR(200), location_id VARCHAR(10), state VARCHAR(2),
    outcome VARCHAR(30), referral_generated BOOLEAN, notes VARCHAR(500),
    duration_minutes INT, miles_driven FLOAT);

CREATE OR REPLACE TABLE PHYSICIAN_REFERRALS (
    referral_id VARCHAR(12), physician_npi VARCHAR(15), physician_name VARCHAR(100),
    physician_specialty VARCHAR(50), facility_name VARCHAR(200), patient_id VARCHAR(15),
    referral_date DATE, equipment_category VARCHAR(50), hcpcs_code VARCHAR(10),
    equipment_name VARCHAR(100), location_id VARCHAR(10), state VARCHAR(2),
    referral_source VARCHAR(50), status VARCHAR(20), days_to_setup INT,
    revenue FLOAT, is_new_physician BOOLEAN);

CREATE OR REPLACE TABLE CMS_RESPIRATORY_CLAIMS (
    cms_claim_id VARCHAR(15), reporting_quarter VARCHAR(10), state VARCHAR(2),
    city VARCHAR(100), zip_code VARCHAR(10), hcpcs_code VARCHAR(10),
    equipment_category VARCHAR(50), beneficiary_count INT, total_claims INT,
    total_allowed FLOAT, total_paid FLOAT, provider_type VARCHAR(50),
    home_health_share BOOLEAN, competitor_count INT);

CREATE OR REPLACE TABLE CALL_DETAIL_RECORDS (
    cdr_id VARCHAR(15), phone_system VARCHAR(20), call_date DATE, call_time TIME,
    direction VARCHAR(10), call_type VARCHAR(30), caller_id VARCHAR(20),
    patient_id VARCHAR(15), agent_id VARCHAR(10), agent_name VARCHAR(100),
    location_id VARCHAR(10), queue_name VARCHAR(30), wait_time_seconds INT,
    handle_time_seconds INT, after_call_work_seconds INT, total_duration_seconds INT,
    abandoned BOOLEAN, transferred BOOLEAN, first_call_resolution BOOLEAN,
    disposition VARCHAR(30), satisfaction_score INT, recording_available BOOLEAN);

CREATE OR REPLACE TABLE CALL_AGENT_PERFORMANCE (
    agent_id VARCHAR(10), agent_name VARCHAR(100), location_id VARCHAR(10),
    phone_system VARCHAR(20), team VARCHAR(30), hire_date DATE,
    avg_handle_time_seconds INT, avg_after_call_work_seconds INT,
    first_call_resolution_rate FLOAT, calls_per_hour FLOAT,
    avg_satisfaction_score FLOAT, adherence_rate FLOAT, utilization_rate FLOAT,
    quality_score FLOAT, escalation_rate FLOAT, is_active BOOLEAN);

CREATE OR REPLACE TABLE PATIENT_SATISFACTION (
    survey_id VARCHAR(12), patient_id VARCHAR(15), survey_date DATE,
    channel VARCHAR(10), interaction_type VARCHAR(30), overall_score INT,
    ease_of_contact INT, agent_knowledge INT, issue_resolved BOOLEAN,
    wait_time_acceptable BOOLEAN, would_recommend BOOLEAN,
    comments VARCHAR(500), location_id VARCHAR(10));

-- Load all CSVs
COPY INTO LOCATIONS               FROM @HOME_HEALTH_DATA_STAGE/locations.csv              FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO PAYER_CONTRACTS         FROM @HOME_HEALTH_DATA_STAGE/payer_contracts.csv        FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO CLAIMS_SUBMISSIONS      FROM @HOME_HEALTH_DATA_STAGE/claims_submissions.csv     FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO CLAIMS_DENIALS          FROM @HOME_HEALTH_DATA_STAGE/claims_denials.csv         FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO DENIAL_APPEALS          FROM @HOME_HEALTH_DATA_STAGE/denial_appeals.csv         FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO SALES_REP_ACTIVITY      FROM @HOME_HEALTH_DATA_STAGE/sales_rep_activity.csv     FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO PHYSICIAN_REFERRALS     FROM @HOME_HEALTH_DATA_STAGE/physician_referrals.csv    FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO CMS_RESPIRATORY_CLAIMS  FROM @HOME_HEALTH_DATA_STAGE/cms_respiratory_claims.csv FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO CALL_DETAIL_RECORDS     FROM @HOME_HEALTH_DATA_STAGE/call_detail_records.csv    FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO CALL_AGENT_PERFORMANCE  FROM @HOME_HEALTH_DATA_STAGE/call_agent_performance.csv FILE_FORMAT = CSV_FORMAT FORCE = TRUE;
COPY INTO PATIENT_SATISFACTION    FROM @HOME_HEALTH_DATA_STAGE/patient_satisfaction.csv   FILE_FORMAT = CSV_FORMAT FORCE = TRUE;

-- Verify row counts
SELECT 'CLAIMS_SUBMISSIONS' AS tbl, COUNT(*) AS rows FROM CLAIMS_SUBMISSIONS
UNION ALL SELECT 'CLAIMS_DENIALS',     COUNT(*) FROM CLAIMS_DENIALS
UNION ALL SELECT 'DENIAL_APPEALS',     COUNT(*) FROM DENIAL_APPEALS
UNION ALL SELECT 'CALL_DETAIL_RECORDS',COUNT(*) FROM CALL_DETAIL_RECORDS
UNION ALL SELECT 'LOCATIONS',          COUNT(*) FROM LOCATIONS
ORDER BY tbl;

## Section 4: HIPAA Governance — Dynamic Data Masking

**One table. Two realities. Zero data copies.**

```
ANALYST role:        patient_id = 'PAT-***MASKED***'
BILLING_ADMIN role:  patient_id = 'PAT-0015473'
```

Snowflake applies masking at **query time** based on your current role — no duplicate tables, no column-level encryption keys to manage.

> **SNOWFLAKE DIFFERENTIATOR vs Fabric:** Microsoft documented 5 min to 2 hour policy propagation delays for column-level security in Fabric.

> **IMPORTANT:** Always run `USE SECONDARY ROLES NONE` before testing masking. Without this, your ACCOUNTADMIN secondary role overrides the masking policy and patient IDs appear unmasked.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;

-- Must unset existing policies before recreating (cannot replace an applied policy)
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id             UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_DENIALS      MODIFY COLUMN patient_id             UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id             UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN referring_physician_npi UNSET MASKING POLICY;

CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_PATIENT_ID AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','BILLING_ADMIN','EXECUTIVE') THEN val
         ELSE 'PAT-***MASKED***' END;

CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_PHONE AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','CALL_CENTER_LEAD','EXECUTIVE') THEN val
         ELSE REGEXP_REPLACE(val, '[0-9]', '*') END;

CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_NPI AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','SALES_MANAGER','BILLING_ADMIN','EXECUTIVE') THEN val
         ELSE LEFT(val,3) || '****' || RIGHT(val,3) END;

ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id              SET MASKING POLICY RAW_DATA.MASK_PATIENT_ID;
ALTER TABLE RAW_DATA.CLAIMS_DENIALS      MODIFY COLUMN patient_id              SET MASKING POLICY RAW_DATA.MASK_PATIENT_ID;
ALTER TABLE RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id              SET MASKING POLICY RAW_DATA.MASK_PHONE;
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN referring_physician_npi SET MASKING POLICY RAW_DATA.MASK_NPI;

CREATE OR REPLACE TAG RAW_DATA.DATA_SENSITIVITY ALLOWED_VALUES 'PHI', 'PII', 'FINANCIAL', 'PUBLIC';
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id    SET TAG RAW_DATA.DATA_SENSITIVITY = 'PHI';
ALTER TABLE RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id     SET TAG RAW_DATA.DATA_SENSITIVITY = 'PII';
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN billed_amount SET TAG RAW_DATA.DATA_SENSITIVITY = 'FINANCIAL';

GRANT SELECT ON ALL TABLES IN SCHEMA RAW_DATA TO ROLE BILLING_ADMIN;
GRANT SELECT ON ALL TABLES IN SCHEMA ANALYTICS TO ROLE BILLING_ADMIN;
GRANT SELECT ON TABLE RAW_DATA.SALES_REP_ACTIVITY    TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.PHYSICIAN_REFERRALS   TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.CMS_RESPIRATORY_CLAIMS TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.CALL_DETAIL_RECORDS   TO ROLE CALL_CENTER_LEAD;
GRANT SELECT ON ALL TABLES IN SCHEMA RAW_DATA TO ROLE ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA ANALYTICS TO ROLE ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA ANALYTICS TO ROLE EXECUTIVE;

SELECT 'Governance policies applied' AS status;

In [ ]:
-- Demonstrate masking: MUST disable secondary roles first
-- (otherwise ACCOUNTADMIN secondary role overrides the masking policy)
USE SECONDARY ROLES NONE;

-- ANALYST: patient_id should be masked
USE ROLE ANALYST;
SELECT claim_id, patient_id, billed_amount FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS LIMIT 5;

In [ ]:
-- BILLING_ADMIN: patient_id should be visible
USE ROLE BILLING_ADMIN;
SELECT claim_id, patient_id, billed_amount FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS LIMIT 5;

-- Always reset secondary roles after testing
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES ALL;

## Section 5: Dynamic Tables — Declarative Medallion Pipeline

**The key differentiator:** You declare WHAT you want — Snowflake handles WHEN and HOW.

```
Bronze (1-min lag)  →  Raw data cleansed as it arrives
Silver (5-min lag)  →  Business KPIs aggregated across all 3 use cases
Gold (DOWNSTREAM)   →  Executive KPIs that refresh when Silver refreshes
Alerts (1-min lag)  →  Real-time denial spikes and call SLA breaches
```

> **vs Fabric:** Requires Data Factory pipelines with explicit trigger logic and activity dependencies.
> **vs Databricks:** Delta Live Tables require notebook configuration per pipeline stage.
> **Snowflake:** `TARGET_LAG = '5 minutes'` — one line.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

-- BRONZE: Cleanse raw data (1-minute lag)
CREATE OR REPLACE DYNAMIC TABLE TRANSFORMED.DT_CLEAN_CLAIMS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT claim_id, patient_id, location_id, UPPER(TRIM(payer_code)) AS payer_code,
       payer_name, UPPER(TRIM(hcpcs_code)) AS hcpcs_code, equipment_name, equipment_category,
       quantity, COALESCE(billed_amount,0) AS billed_amount,
       COALESCE(allowed_amount,0) AS allowed_amount, COALESCE(paid_amount,0) AS paid_amount,
       submission_date, service_date, adjudication_date,
       DATEDIFF(day, submission_date, COALESCE(adjudication_date, CURRENT_DATE())) AS days_in_ar,
       CASE WHEN claim_status='PAID' AND paid_amount>0 THEN TRUE ELSE FALSE END AS is_clean_claim,
       CASE WHEN claim_status='DENIED' THEN TRUE ELSE FALSE END AS is_denied
FROM RAW_DATA.CLAIMS_SUBMISSIONS WHERE claim_id IS NOT NULL;

CREATE OR REPLACE DYNAMIC TABLE TRANSFORMED.DT_CLEAN_CALL_RECORDS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT cdr_id, UPPER(TRIM(phone_system)) AS phone_system, call_date, call_time,
       UPPER(TRIM(direction)) AS direction, INITCAP(TRIM(call_type)) AS call_type,
       patient_id, agent_id, agent_name, location_id, queue_name,
       COALESCE(wait_time_seconds,0) AS wait_time_seconds,
       COALESCE(handle_time_seconds,0) AS handle_time_seconds,
       COALESCE(after_call_work_seconds,0) AS after_call_work_seconds,
       abandoned, transferred, first_call_resolution, disposition, satisfaction_score,
       CASE WHEN wait_time_seconds<=20 THEN 'WITHIN_SLA'
            WHEN wait_time_seconds<=60 THEN 'NEAR_SLA'
            ELSE 'SLA_BREACH' END AS sla_status
FROM RAW_DATA.CALL_DETAIL_RECORDS WHERE cdr_id IS NOT NULL;

-- SILVER: Business KPIs (5-minute lag)
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_DENIAL_RATE_BY_PAYER
TARGET_LAG = '5 minutes' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT payer_code, payer_name, DATE_TRUNC('week', submission_date) AS week_start,
       COUNT(*) AS total_claims,
       SUM(CASE WHEN is_denied THEN 1 ELSE 0 END) AS denied_claims,
       ROUND(SUM(CASE WHEN is_denied THEN 1 ELSE 0 END)*100.0/NULLIF(COUNT(*),0),2) AS denial_rate_pct,
       ROUND(SUM(CASE WHEN is_clean_claim THEN 1 ELSE 0 END)*100.0/NULLIF(COUNT(*),0),2) AS clean_claim_rate_pct,
       SUM(billed_amount) AS total_billed, SUM(paid_amount) AS total_paid,
       ROUND(AVG(days_in_ar),1) AS avg_days_in_ar
FROM TRANSFORMED.DT_CLEAN_CLAIMS
GROUP BY payer_code, payer_name, DATE_TRUNC('week', submission_date);

CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_CALL_CENTER_METRICS
TARGET_LAG = '5 minutes' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT phone_system, location_id, call_type, DATE_TRUNC('day', call_date) AS call_day,
       COUNT(*) AS total_calls,
       COUNT(CASE WHEN abandoned THEN 1 END) AS abandoned_calls,
       ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),2) AS abandonment_rate_pct,
       ROUND(AVG(wait_time_seconds),1) AS avg_wait_seconds,
       ROUND(AVG(handle_time_seconds),1) AS avg_handle_seconds,
       ROUND(COUNT(CASE WHEN first_call_resolution THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END),0),2) AS fcr_rate_pct,
       ROUND(COUNT(CASE WHEN sla_status='WITHIN_SLA' THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN direction='INBOUND' THEN 1 END),0),2) AS service_level_pct
FROM TRANSFORMED.DT_CLEAN_CALL_RECORDS
GROUP BY phone_system, location_id, call_type, DATE_TRUNC('day', call_date);

-- GOLD: Executive KPIs (DOWNSTREAM — refreshes when Silver refreshes)
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_EXECUTIVE_KPIS
TARGET_LAG = DOWNSTREAM WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT 'Q1 2026' AS reporting_period, CURRENT_TIMESTAMP() AS last_refresh,
    (SELECT ROUND(AVG(denial_rate_pct),2) FROM ANALYTICS.DT_DENIAL_RATE_BY_PAYER) AS overall_denial_rate_pct,
    (SELECT ROUND(AVG(clean_claim_rate_pct),2) FROM ANALYTICS.DT_DENIAL_RATE_BY_PAYER) AS clean_claim_rate_pct,
    (SELECT SUM(total_calls) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS total_calls,
    (SELECT ROUND(AVG(abandonment_rate_pct),2) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS avg_abandonment_pct,
    (SELECT ROUND(AVG(fcr_rate_pct),2) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS avg_fcr_rate_pct;

-- ALERTS: Real-time denial spikes (1-minute lag)
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_REALTIME_ALERTS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT 'DENIAL_SPIKE' AS alert_type, 'HIGH' AS severity, location_id,
       'Denial spike: ' || COUNT(*) || ' denials in last 7 days' AS alert_message,
       COUNT(*) AS metric_value, CURRENT_TIMESTAMP() AS alert_timestamp
FROM TRANSFORMED.DT_CLEAN_CLAIMS
WHERE is_denied AND submission_date >= DATEADD(day,-7,CURRENT_DATE())
GROUP BY location_id HAVING COUNT(*) > 15
UNION ALL
SELECT 'CALL_SLA_BREACH', 'CRITICAL', location_id,
       'SLA breach: ' || ROUND(COUNT(CASE WHEN sla_status='SLA_BREACH' THEN 1 END)*100.0/COUNT(*),1) || '% above threshold',
       ROUND(COUNT(CASE WHEN sla_status='SLA_BREACH' THEN 1 END)*100.0/NULLIF(COUNT(*),0),1),
       CURRENT_TIMESTAMP()
FROM TRANSFORMED.DT_CLEAN_CALL_RECORDS
WHERE call_date >= DATEADD(day,-1,CURRENT_DATE()) AND direction='INBOUND'
GROUP BY location_id
HAVING ROUND(COUNT(CASE WHEN sla_status='SLA_BREACH' THEN 1 END)*100.0/NULLIF(COUNT(*),0),1) > 20;

SHOW DYNAMIC TABLES IN DATABASE HOME_HEALTH_DEMO;
SELECT 'Dynamic Tables pipeline created (Bronze/Silver/Gold/Alerts)' AS status;

## Section 6: Domain Analytics — Three Use Case Deep Dives

### Use Case 1: Denials Reduction
DME denial rates average 15-20% industy-wide. For a company billing $400M/year, a 12% denial rate = $48M at risk per year.

### Use Case 2: Sales Data Analysis
Blend internal CRM with **CMS Medicare market data** (from Snowflake Marketplace) to reveal true market penetration — which states and territories have the most upside.

### Use Case 3: Call Center Consolidation
Three disparate phone systems (Avaya, Five9, RingCentral) normalized into one analytics layer. Cross-domain insight: **denied claims directly drive billing inquiry call volume** — visible only when all data is on one platform.

In [ ]:
-- USE CASE 1: DENIALS ANALYTICS
USE SCHEMA HOME_HEALTH_DEMO.ANALYTICS;

-- Overall denial rate Q1 2026
SELECT ROUND(COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct,
       COUNT(CASE WHEN claim_status='DENIED' THEN 1 END) AS denied_count,
       COUNT(*) AS total_claims,
       SUM(CASE WHEN claim_status='DENIED' THEN billed_amount ELSE 0 END) AS revenue_at_risk
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS;

-- Root cause Pareto (Pareto principle: top 3 causes = 80% of denials)
SELECT root_cause, denial_category, COUNT(*) AS denial_count,
       SUM(denied_amount) AS total_denied,
       ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER (),1) AS pct_of_total,
       SUM(COUNT(*)*100.0/SUM(COUNT(*)) OVER ()) OVER (ORDER BY COUNT(*) DESC) AS running_pct
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS
GROUP BY root_cause, denial_category ORDER BY denial_count DESC LIMIT 10;

-- Denial rate by payer (identify worst payers)
SELECT payer_name, COUNT(*) AS total_claims,
       COUNT(CASE WHEN claim_status='DENIED' THEN 1 END) AS denied,
       ROUND(COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS
GROUP BY payer_name ORDER BY denial_rate_pct DESC;

In [ ]:
-- USE CASE 2: SALES INTELLIGENCE
-- Market penetration: internal CRM + CMS Medicare market data
SELECT cms.state,
       SUM(cms.total_claims) AS total_market_claims,
       SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END) AS our_claims,
       ROUND(SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END)*100.0/NULLIF(SUM(cms.total_claims),0),2) AS market_share_pct,
       CASE WHEN ROUND(SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END)*100.0/NULLIF(SUM(cms.total_claims),0),2) >= 35 THEN 'Well Penetrated'
            WHEN ROUND(SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END)*100.0/NULLIF(SUM(cms.total_claims),0),2) >= 20 THEN 'Growth Opportunity'
            ELSE 'Expansion Target' END AS market_status
FROM HOME_HEALTH_DEMO.RAW_DATA.CMS_RESPIRATORY_CLAIMS cms
GROUP BY cms.state ORDER BY market_share_pct DESC LIMIT 15;

-- Rep leaderboard
SELECT rep_name, territory, COUNT(*) AS total_activities,
       COUNT(CASE WHEN referral_generated THEN 1 END) AS referrals,
       ROUND(COUNT(CASE WHEN referral_generated THEN 1 END)*100.0/COUNT(*),1) AS conversion_rate_pct,
       COUNT(DISTINCT physician_npi) AS physicians_engaged
FROM HOME_HEALTH_DEMO.RAW_DATA.SALES_REP_ACTIVITY
GROUP BY rep_name, territory ORDER BY referrals DESC LIMIT 15;

In [ ]:
-- USE CASE 3: CALL CENTER CONSOLIDATION
-- Three phone systems in one query (the "before Snowflake" answer took 3 different BI tools)
SELECT phone_system, COUNT(*) AS total_calls,
       ROUND(AVG(handle_time_seconds),1) AS avg_handle_secs,
       ROUND(AVG(wait_time_seconds),1) AS avg_wait_secs,
       ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),2) AS abandonment_rate_pct,
       ROUND(COUNT(CASE WHEN first_call_resolution THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END),0),2) AS fcr_rate_pct,
       ROUND(AVG(satisfaction_score),2) AS avg_csat,
       COUNT(DISTINCT agent_id) AS agents
FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS
GROUP BY phone_system ORDER BY total_calls DESC;

-- Cross-domain: denied claims driving billing call spikes (ONLY possible with unified platform)
SELECT l.location_id, l.state, l.region,
       COALESCE(d.denial_count, 0) AS denial_count,
       COALESCE(c.billing_calls, 0) AS billing_calls,
       CASE WHEN COALESCE(d.denial_count,0)>20 AND COALESCE(c.billing_calls,0)>50 THEN 'HIGH_RISK'
            WHEN COALESCE(d.denial_count,0)>10 OR COALESCE(c.billing_calls,0)>25 THEN 'MEDIUM_RISK'
            ELSE 'LOW_RISK' END AS risk_category
FROM HOME_HEALTH_DEMO.RAW_DATA.LOCATIONS l
LEFT JOIN (SELECT location_id, COUNT(*) AS denial_count FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS GROUP BY 1) d ON l.location_id=d.location_id
LEFT JOIN (SELECT location_id, COUNT(CASE WHEN call_type='Billing Inquiry' THEN 1 END) AS billing_calls FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS GROUP BY 1) c ON l.location_id=c.location_id
WHERE l.is_active=TRUE ORDER BY denial_count DESC LIMIT 15;

## Section 7: Semantic View — Natural Language to SQL (Cortex Analyst)

A Semantic View is a **native SQL object** that maps business terminology to your tables. Cortex Analyst uses it to answer natural language questions with accurate SQL automatically.

> **vs Fabric:** Power BI semantic model is a separate tool requiring Power BI Desktop + publishing to cloud.
> **vs Databricks:** No equivalent native concept — must write custom LLM prompts or use external BI tools.
> **Snowflake:** `CREATE SEMANTIC VIEW` in SQL — version-controlled like any other database object.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA HOME_HEALTH_DEMO.ANALYTICS;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

CREATE OR REPLACE SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS
  TABLES (
    CLAIMS_SUBMISSIONS AS HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS
      PRIMARY KEY (claim_id) WITH SYNONYMS = ('claims', 'submissions', 'billing')
      COMMENT = 'DME claims submitted to payers Q1 2026',
    CLAIMS_DENIALS AS HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS
      PRIMARY KEY (denial_id) WITH SYNONYMS = ('denials', 'rejected claims', 'denied')
      COMMENT = 'Claims denied by payers with CARC codes',
    DENIAL_APPEALS AS HOME_HEALTH_DEMO.RAW_DATA.DENIAL_APPEALS
      PRIMARY KEY (appeal_id) WITH SYNONYMS = ('appeals', 'rework', 'overturned')
      COMMENT = 'Appeals filed against denied claims',
    LOCATIONS AS HOME_HEALTH_DEMO.RAW_DATA.LOCATIONS
      PRIMARY KEY (location_id) WITH SYNONYMS = ('centers', 'branches', 'offices', 'sites')
      COMMENT = 'Operating centers across 48 states',
    SALES_REP_ACTIVITY AS HOME_HEALTH_DEMO.RAW_DATA.SALES_REP_ACTIVITY
      PRIMARY KEY (activity_id) WITH SYNONYMS = ('sales', 'rep visits', 'outreach', 'meetings')
      COMMENT = 'Sales rep activities and physician contacts',
    CALL_DETAIL_RECORDS AS HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS
      PRIMARY KEY (cdr_id) WITH SYNONYMS = ('calls', 'CDR', 'phone calls')
      COMMENT = 'Call detail records from Avaya, Five9, and RingCentral'
  )
  RELATIONSHIPS (
    denials_to_claims    AS CLAIMS_DENIALS (claim_id)        REFERENCES CLAIMS_SUBMISSIONS (claim_id),
    appeals_to_denials   AS DENIAL_APPEALS (denial_id)       REFERENCES CLAIMS_DENIALS (denial_id),
    claims_to_locations  AS CLAIMS_SUBMISSIONS (location_id) REFERENCES LOCATIONS (location_id),
    denials_to_locations AS CLAIMS_DENIALS (location_id)     REFERENCES LOCATIONS (location_id),
    calls_to_locations   AS CALL_DETAIL_RECORDS (location_id) REFERENCES LOCATIONS (location_id)
  )
  DIMENSIONS (
    CLAIMS_SUBMISSIONS.payer_name_dim       AS payer_name       COMMENT = 'Payer Name',
    CLAIMS_SUBMISSIONS.equipment_category_dim AS equipment_category COMMENT = 'Equipment Category',
    CLAIMS_SUBMISSIONS.claim_status_dim     AS claim_status     COMMENT = 'Claim Status',
    CLAIMS_SUBMISSIONS.submission_date_dim  AS submission_date  COMMENT = 'Submission Date',
    CLAIMS_DENIALS.denial_code_dim          AS denial_code      COMMENT = 'Denial Code (CARC)',
    CLAIMS_DENIALS.root_cause_dim           AS root_cause       COMMENT = 'Root Cause of Denial',
    LOCATIONS.state_dim                     AS state            COMMENT = 'State',
    LOCATIONS.region_dim                    AS region           COMMENT = 'Region',
    SALES_REP_ACTIVITY.rep_name_dim         AS rep_name         COMMENT = 'Sales Rep Name',
    CALL_DETAIL_RECORDS.phone_system_dim    AS phone_system     COMMENT = 'Phone System',
    CALL_DETAIL_RECORDS.call_type_dim       AS call_type        COMMENT = 'Call Type',
    CALL_DETAIL_RECORDS.call_date_dim       AS call_date        COMMENT = 'Call Date'
  )
  METRICS (
    CLAIMS_SUBMISSIONS.denial_rate AS
      COUNT(CASE WHEN claim_status = 'DENIED' THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Percentage of claims denied',
    CLAIMS_SUBMISSIONS.clean_claim_rate AS
      COUNT(CASE WHEN claim_status = 'PAID' AND paid_amount > 0 THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Percentage paid on first submission',
    CLAIMS_SUBMISSIONS.avg_days_in_ar AS
      AVG(DATEDIFF(day, submission_date, adjudication_date))
      COMMENT = 'Average days from submission to payment',
    CLAIMS_DENIALS.total_denied_amount AS SUM(denied_amount)
      COMMENT = 'Total dollar amount denied',
    CALL_DETAIL_RECORDS.abandonment_rate AS
      COUNT(CASE WHEN abandoned THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Call abandonment rate',
    CALL_DETAIL_RECORDS.fcr_rate AS
      COUNT(CASE WHEN first_call_resolution THEN 1 END) * 100.0 / NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END), 0)
      COMMENT = 'First call resolution rate',
    SALES_REP_ACTIVITY.conversion_rate AS
      COUNT(CASE WHEN referral_generated THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Sales activity to referral conversion rate'
  )
  COMMENT = 'Semantic view for Home Health DME - denials, sales, call center Q1 2026'
  AI_VERIFIED_QUERIES (
    overall_denial_rate AS (
      QUESTION 'What is the overall denial rate?'
      SQL 'SELECT ROUND(COUNT(CASE WHEN claim_status=''DENIED'' THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS'
    ),
    denial_rate_by_payer AS (
      QUESTION 'Show denial rate by payer sorted highest to lowest'
      SQL 'SELECT payer_name, ROUND(COUNT(CASE WHEN claim_status=''DENIED'' THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct, COUNT(*) AS total_claims FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS GROUP BY payer_name ORDER BY denial_rate_pct DESC'
    ),
    top_denial_reasons AS (
      QUESTION 'What are the top denial reasons?'
      SQL 'SELECT denial_code, denial_reason, root_cause, COUNT(*) AS count, SUM(denied_amount) AS total_denied FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS GROUP BY denial_code, denial_reason, root_cause ORDER BY count DESC LIMIT 10'
    ),
    call_abandonment_by_system AS (
      QUESTION 'What is the call abandonment rate by phone system?'
      SQL 'SELECT phone_system, ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/COUNT(*),2) AS abandonment_rate, COUNT(*) AS total_calls FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS GROUP BY phone_system'
    ),
    top_sales_reps AS (
      QUESTION 'Which sales reps generated the most referrals this quarter?'
      SQL 'SELECT rep_name, territory, COUNT(CASE WHEN referral_generated THEN 1 END) AS referrals, ROUND(COUNT(CASE WHEN referral_generated THEN 1 END)*100.0/COUNT(*),2) AS conversion_rate FROM HOME_HEALTH_DEMO.RAW_DATA.SALES_REP_ACTIVITY GROUP BY rep_name, territory ORDER BY referrals DESC LIMIT 10'
    )
  );

GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE BILLING_ADMIN;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE SALES_MANAGER;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE ANALYST;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE EXECUTIVE;

SELECT '✅ Semantic View ready: HOME_HEALTH_DEMO.ANALYTICS.SV_HOME_HEALTH_OPERATIONS' AS status;

## Section 8: Cortex Search — RAG Over Policy Documents

### Upload 10 Policy Documents First
```sql
-- From the cloned repo, upload the documents/ folder:
PUT file://./documents/01_claims_submission_guidelines.md    @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/02_denial_appeal_procedures.md        @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/03_medicare_cmn_requirements.md       @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/04_equipment_authorization_policy.md  @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/05_call_center_sla_standards.md       @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/06_sales_territory_policy.md          @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/07_hipaa_compliance_dme.md            @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/08_payer_billing_rules.md             @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/09_quality_metrics_definitions.md     @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/10_referral_management_guidelines.md  @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
```

> **vs Fabric:** Azure AI Search + Azure OpenAI + custom indexer = 3 Azure services + weeks of work.
> **vs Databricks:** Vector Search + embedding model + serving endpoint = separate services.
> **Snowflake:** One `CREATE CORTEX SEARCH SERVICE` DDL.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA HOME_HEALTH_DEMO.DOCUMENTS;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

CREATE OR REPLACE TABLE HOME_HEALTH_POLICY_DOCUMENTS (
    document_id     VARCHAR(20), document_title VARCHAR(500),
    policy_number   VARCHAR(20), department VARCHAR(100),
    document_type   VARCHAR(50), effective_date DATE,
    chunk_id INT,   chunk_content TEXT
);

CREATE OR REPLACE TABLE HOME_HEALTH_RAW_DOCUMENTS (
    filename VARCHAR(500), file_content VARCHAR(16777216)
);

COPY INTO HOME_HEALTH_RAW_DOCUMENTS (filename, file_content)
FROM (SELECT METADATA$FILENAME, TO_VARCHAR($1) FROM @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE/)
FILE_FORMAT = (TYPE='CSV' FIELD_DELIMITER=NONE RECORD_DELIMITER=NONE ESCAPE_UNENCLOSED_FIELD=NONE)
FORCE = TRUE;

SELECT filename, LENGTH(file_content) AS chars FROM HOME_HEALTH_RAW_DOCUMENTS;

In [ ]:
-- Chunk docs and load into search table
INSERT INTO HOME_HEALTH_POLICY_DOCUMENTS
WITH doc_metadata AS (
    SELECT filename, file_content,
        CASE WHEN filename LIKE '%01_claims%'    THEN 'CLM-001'
             WHEN filename LIKE '%02_denial%'    THEN 'DEN-001'
             WHEN filename LIKE '%03_medicare%'  THEN 'CMN-001'
             WHEN filename LIKE '%04_equipment%' THEN 'EQP-001'
             WHEN filename LIKE '%05_call%'      THEN 'CC-001'
             WHEN filename LIKE '%06_sales%'     THEN 'SLS-001'
             WHEN filename LIKE '%07_hipaa%'     THEN 'HIP-001'
             WHEN filename LIKE '%08_payer%'     THEN 'PAY-001'
             WHEN filename LIKE '%09_quality%'   THEN 'QM-001'
             WHEN filename LIKE '%10_referral%'  THEN 'REF-001' END AS policy_number,
        CASE WHEN filename LIKE '%01_claims%'    THEN 'Claims Submission Guidelines'
             WHEN filename LIKE '%02_denial%'    THEN 'Denial Appeal Procedures'
             WHEN filename LIKE '%03_medicare%'  THEN 'Medicare CMN Requirements'
             WHEN filename LIKE '%04_equipment%' THEN 'Equipment Authorization Policy'
             WHEN filename LIKE '%05_call%'      THEN 'Call Center SLA Standards'
             WHEN filename LIKE '%06_sales%'     THEN 'Sales Territory Policy'
             WHEN filename LIKE '%07_hipaa%'     THEN 'HIPAA Compliance for DME'
             WHEN filename LIKE '%08_payer%'     THEN 'Payer-Specific Billing Rules'
             WHEN filename LIKE '%09_quality%'   THEN 'Quality Metrics Definitions'
             WHEN filename LIKE '%10_referral%'  THEN 'Referral Management Guidelines' END AS document_title
    FROM HOME_HEALTH_RAW_DOCUMENTS
),
chunked AS (
    SELECT policy_number, document_title,
        SUBSTRING(file_content,1,2000) AS chunk_1,
        CASE WHEN LEN(file_content)>2000 THEN SUBSTRING(file_content,2001,2000) END AS chunk_2,
        CASE WHEN LEN(file_content)>4000 THEN SUBSTRING(file_content,4001,2000) END AS chunk_3,
        CASE WHEN LEN(file_content)>6000 THEN SUBSTRING(file_content,6001,2000) END AS chunk_4,
        CASE WHEN LEN(file_content)>8000 THEN SUBSTRING(file_content,8001,2000) END AS chunk_5
    FROM doc_metadata
)
SELECT policy_number||'-1', document_title, policy_number, 'Operations', 'Policy', '2026-01-01', 1, chunk_1 FROM chunked WHERE chunk_1 IS NOT NULL
UNION ALL SELECT policy_number||'-2', document_title, policy_number, 'Operations', 'Policy', '2026-01-01', 2, chunk_2 FROM chunked WHERE chunk_2 IS NOT NULL
UNION ALL SELECT policy_number||'-3', document_title, policy_number, 'Operations', 'Policy', '2026-01-01', 3, chunk_3 FROM chunked WHERE chunk_3 IS NOT NULL
UNION ALL SELECT policy_number||'-4', document_title, policy_number, 'Operations', 'Policy', '2026-01-01', 4, chunk_4 FROM chunked WHERE chunk_4 IS NOT NULL
UNION ALL SELECT policy_number||'-5', document_title, policy_number, 'Operations', 'Policy', '2026-01-01', 5, chunk_5 FROM chunked WHERE chunk_5 IS NOT NULL;

CREATE OR REPLACE CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH
    ON chunk_content WAREHOUSE = HOME_HEALTH_ANALYTICS_WH TARGET_LAG = '1 hour'
    AS (SELECT document_id, document_title, policy_number, department, document_type, effective_date, chunk_id, chunk_content
        FROM HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_DOCUMENTS);

GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE BILLING_ADMIN;
GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE ANALYST;
GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE EXECUTIVE;

SELECT '✅ Cortex Search Service: HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH' AS status;

## Section 9: Snowflake Intelligence Agent

**Agentic AI** that combines Cortex Analyst (structured data) + Cortex Search (documents) into one unified assistant.

| Tool | Queries |
|------|---------|
| `home_health_data` | Claims, denials, sales, call center (Semantic View) |
| `policy_search` | 10 policy documents — CMN requirements, SLA targets, billing rules |

### Sample Questions to Try
- *"What is our denial rate vs the 5% policy target?"* → uses both tools
- *"Which phone system has the worst abandonment rate?"* → Cortex Analyst
- *"What documentation do I need to appeal a CO-16 denial?"* → Cortex Search
- *"What is the timely filing limit for Medicare?"* → Cortex Search

> **vs Fabric:** Copilot Studio + Azure AI Search + Azure OpenAI + Power BI connector = weeks of integration.
> **vs Databricks:** Mosaic AI Agent Framework + Vector Search + model serving = multiple services.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS SNOWFLAKE_INTELLIGENCE;
GRANT USAGE ON DATABASE SNOWFLAKE_INTELLIGENCE TO ROLE PUBLIC;
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_INTELLIGENCE.AGENTS;
GRANT USAGE ON SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS TO ROLE PUBLIC;
GRANT CREATE AGENT ON SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS TO ROLE ACCOUNTADMIN;

CREATE OR REPLACE AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT
    COMMENT = 'Home Health DME Operations Intelligence Agent'
    PROFILE = '{"display_name": "Home Health Operations Assistant", "color": "#0066CC"}'
    FROM SPECIFICATION $$
    {
        "models": {"orchestration": "claude-4-sonnet"},
        "instructions": {
            "orchestration": "You are a Home Health DME operations assistant. Use home_health_data for structured queries about claims, denials, sales, and call center metrics. Use policy_search for policies, procedures, and compliance requirements. Combine both when questions involve performance vs targets.",
            "response": "Provide clear, actionable insights. Reference policy numbers. Format numbers as currency or percentages. For denial questions, always mention root cause and recommended next action.",
            "system": "You assist billing administrators, sales managers, and call center leads at a DME provider. Maintain HIPAA compliance — never expose individual patient identifiers."
        },
        "tools": [
            {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "home_health_data",
              "description": "Query DME operational data: claims, denials, appeals, sales activities, physician referrals, CMS market data, call records, agent performance."}},
            {"tool_spec": {"type": "cortex_search", "name": "policy_search",
              "description": "Search DME policy documents: claims submission, denial appeals, Medicare CMN, call center SLAs, sales policy, HIPAA compliance, payer billing rules, quality metrics."}}
        ],
        "tool_resources": {
            "home_health_data": {
                "semantic_view": "HOME_HEALTH_DEMO.ANALYTICS.SV_HOME_HEALTH_OPERATIONS",
                "execution_environment": {"type": "warehouse", "warehouse": "HOME_HEALTH_ANALYTICS_WH"},
                "query_timeout": 120
            },
            "policy_search": {
                "search_service": "HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH",
                "max_results": 10,
                "columns": ["chunk_content", "document_title", "policy_number", "department"]
            }
        },
        "sample_prompts": [
            "What is our overall denial rate for Q1 2026?",
            "Show me denial rates by payer, sorted highest to lowest",
            "What is our denial rate vs the 5% policy target?",
            "What are the CMN requirements for oxygen equipment?",
            "Which locations have the highest denial rates?",
            "What is our call abandonment rate by phone system?",
            "What documentation do I need to appeal a CO-16 denial?",
            "What is the SLA target for the billing queue?",
            "Which sales reps generated the most referrals?",
            "What is the timely filing limit for Medicare claims?"
        ]
    }
    $$;

GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE PUBLIC;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE BILLING_ADMIN;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE SALES_MANAGER;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE ANALYST;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE EXECUTIVE;

SHOW AGENTS IN SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS;
SELECT 'Agent created: SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT' AS status;
SELECT 'Open at: AI & ML > Snowflake Intelligence in Snowsight' AS instructions;

## Section 10: FHIR R4 Pipeline — EHR Data via VARIANT + LATERAL FLATTEN

### What is FHIR?
FHIR (Fast Healthcare Interoperability Resources) R4 is the HL7 standard for EHR data exchange. The `data/home_health_fhir_bundle.json` file contains 500 FHIR resources for 50 patients — **all 50 are real patients from claims_denials.csv** so the cross-use-case join works.

| Resource | Count | Code System | DME Relevance |
|----------|-------|-------------|---------------|
| Patient | 50 | — | Demographics, Medicare ID |
| Condition | 100 | ICD-10, SNOMED | OSA, COPD — diagnoses behind equipment orders |
| Observation | 150 | LOINC | SpO2, AHI — clinical justification for O2/CPAP |
| MedicationRequest | 100 | RxNorm + HCPCS | Equipment prescriptions |
| Encounter | 100 | SNOMED | Home visits, telehealth encounters |

### The Snowflake Advantage
```sql
-- Entire nested FHIR bundle loaded in one statement — no schema mapping
COPY INTO FHIR_BUNDLES FROM (SELECT METADATA$FILENAME, PARSE_JSON($1) FROM @stage)
FILE_FORMAT = (TYPE='JSON') PATTERN='.*\.json';

-- Extract any resource type on demand — no pipeline configuration
SELECT res.value:resource:name[0]:family::VARCHAR FROM FHIR_BUNDLES b,
LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Patient';
```

> **vs Fabric:** Requires Data Factory + custom JSON schema mapping per resource type.

### Upload FHIR Bundle First
```sql
PUT file://./data/home_health_fhir_bundle.json @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
```

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

-- Raw FHIR table — entire bundle stored as VARIANT (no schema needed!)
CREATE OR REPLACE TABLE HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES (
    bundle_id   VARCHAR,
    source_file VARCHAR,
    loaded_at   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    bundle_data VARIANT
);

-- Load JSON from stage — PATTERN filters to .json only (not CSVs on same stage)
COPY INTO HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES (source_file, bundle_data)
FROM (
    SELECT METADATA$FILENAME, PARSE_JSON($1)
    FROM @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DATA_STAGE
)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = FALSE)
PATTERN = '.*\.json'
FORCE = TRUE;

-- Verify: LATERAL FLATTEN shows all resource types
SELECT res.value:resource:resourceType::VARCHAR AS resource_type, COUNT(*) AS count
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
GROUP BY resource_type ORDER BY count DESC;

In [ ]:
USE SCHEMA HOME_HEALTH_DEMO.FHIR_ANALYTICS;

-- Patient view
CREATE OR REPLACE VIEW V_FHIR_PATIENT AS
SELECT
    res.value:resource:id::VARCHAR                         AS fhir_patient_id,
    res.value:resource:identifier[0]:value::VARCHAR        AS home_health_patient_id,
    res.value:resource:name[0]:family::VARCHAR             AS family_name,
    res.value:resource:name[0]:given[0]::VARCHAR           AS given_name,
    res.value:resource:gender::VARCHAR                     AS gender,
    res.value:resource:birthDate::DATE                     AS birth_date,
    DATEDIFF(year, res.value:resource:birthDate::DATE, CURRENT_DATE()) AS age_years,
    res.value:resource:address[0]:state::VARCHAR           AS state
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Patient';

-- Condition view
CREATE OR REPLACE VIEW V_FHIR_CONDITION AS
SELECT
    res.value:resource:id::VARCHAR                                        AS condition_id,
    SPLIT_PART(res.value:resource:subject:reference::VARCHAR, '/', -1)    AS fhir_patient_id,
    res.value:resource:code:coding[1]:code::VARCHAR                       AS icd10_code,
    res.value:resource:code:coding[1]:display::VARCHAR                    AS icd10_display,
    res.value:resource:code:coding[0]:code::VARCHAR                       AS snomed_code,
    res.value:resource:onsetDateTime::TIMESTAMP                           AS onset_datetime
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Condition';

-- Observation view (SpO2, AHI, vitals)
CREATE OR REPLACE VIEW V_FHIR_OBSERVATION AS
SELECT
    res.value:resource:id::VARCHAR                                        AS observation_id,
    SPLIT_PART(res.value:resource:subject:reference::VARCHAR, '/', -1)    AS fhir_patient_id,
    res.value:resource:code:coding[0]:code::VARCHAR                       AS loinc_code,
    res.value:resource:code:coding[0]:display::VARCHAR                    AS observation_name,
    res.value:resource:valueQuantity:value::FLOAT                         AS value,
    res.value:resource:valueQuantity:unit::VARCHAR                        AS unit,
    res.value:resource:effectiveDateTime::TIMESTAMP                       AS effective_datetime
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Observation';

-- Cross-use-case: FHIR clinical profile joined to denied claims
-- ONLY possible because all data is on one unified platform
CREATE OR REPLACE VIEW V_DENIED_PATIENTS_CLINICAL_PROFILE AS
SELECT p.home_health_patient_id, p.given_name, p.family_name, p.age_years, p.gender, p.state,
       cond.icd10_display AS primary_diagnosis, cond.icd10_code,
       d.denial_code, d.denial_reason, d.root_cause AS denial_root_cause,
       d.denied_amount, d.status AS denial_status
FROM HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_PATIENT p
JOIN HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_CONDITION cond ON cond.fhir_patient_id = p.fhir_patient_id
JOIN HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS d ON d.patient_id = p.home_health_patient_id;

SELECT '✅ FHIR R4 Pipeline complete' AS status;

In [ ]:
-- Validate cross-use-case join: clinical diagnoses of patients with denied claims
SELECT icd10_code, icd10_display, denial_root_cause,
       COUNT(*) AS patient_count, SUM(denied_amount) AS denied_revenue
FROM HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_DENIED_PATIENTS_CLINICAL_PROFILE
GROUP BY icd10_code, icd10_display, denial_root_cause
ORDER BY denied_revenue DESC;

-- SpO2 readings by state (clinical quality insight)
SELECT p.state, ROUND(AVG(o.value),1) AS avg_spo2, COUNT(DISTINCT p.fhir_patient_id) AS patients
FROM HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_OBSERVATION o
JOIN HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_PATIENT p ON o.fhir_patient_id = p.fhir_patient_id
WHERE o.loinc_code IN ('59408-5', '2708-6')
GROUP BY p.state ORDER BY avg_spo2;

## Section 11: Streamlit Dashboard — Deploy in Snowflake

1. Navigate to **Projects > Streamlit > + Streamlit App** in Snowsight
2. Configure:
   - **Name:** `Home_Health_Operations_Dashboard`
   - **Database:** `HOME_HEALTH_DEMO` | **Schema:** `ANALYTICS`
   - **Warehouse:** `HOME_HEALTH_ANALYTICS_WH`
3. Paste the code from `home_health_analytics_app_sis.py`
4. Add package: `plotly`
5. Click **Run**

| Tab | Content |
|-----|---------|
| Executive Summary | Revenue at risk, denial rate, cross-use-case KPIs |
| Denials Command Center | Weekly trend, payer breakdown, root cause Pareto |
| Sales Intelligence | Market penetration, rep leaderboard, CMS comparison |
| Call Center Operations | Avaya / Five9 / RingCentral unified metrics |
| AI Assistant | Embedded Cortex Intelligence chatbot |

In [ ]:
-- Grant Streamlit access after deploying
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE BILLING_ADMIN;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE SALES_MANAGER;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE ANALYST;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE EXECUTIVE;

SELECT '✅ Home Health SnowDay HOL Complete!' AS status;
SELECT feature FROM (VALUES
  ('1. Dynamic Tables - Bronze/Silver/Gold medallion pipeline (TARGET_LAG, DOWNSTREAM)'),
  ('2. Cortex Analyst - Natural language to SQL via Semantic View'),
  ('3. Cortex Search - RAG over 10 DME policy documents'),
  ('4. Snowflake Intelligence Agent - Analyst + Search combined'),
  ('5. Streamlit in Snowflake - 5-tab dashboard with embedded AI'),
  ('6. Snowflake Marketplace - CMS Medicare data (zero ETL)'),
  ('7. Dynamic Data Masking - PHI protection by role (HIPAA)'),
  ('8. Row Access Policies - Location-based data filtering'),
  ('9. Semi-Structured Data (VARIANT) - FHIR R4 bundle ingestion'),
  ('10. LATERAL FLATTEN - Nested JSON to relational views'),
  ('11. Multi-Cluster Warehouses - Auto-scale, per-second billing'),
  ('12. Data Classification Tags - PHI/PII/FINANCIAL tagging')
) AS t(feature);

## Section 12: Cleanup

Run this cell to tear down all HOL objects.

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP DATABASE IF EXISTS HOME_HEALTH_DEMO;
DROP DATABASE IF EXISTS SNOWFLAKE_INTELLIGENCE;
DROP WAREHOUSE IF EXISTS HOME_HEALTH_LOAD_WH;
DROP WAREHOUSE IF EXISTS HOME_HEALTH_ANALYTICS_WH;
DROP WAREHOUSE IF EXISTS HOME_HEALTH_ADHOC_WH;
DROP ROLE IF EXISTS BILLING_ADMIN;
DROP ROLE IF EXISTS SALES_MANAGER;
DROP ROLE IF EXISTS CALL_CENTER_LEAD;
DROP ROLE IF EXISTS ANALYST;
DROP ROLE IF EXISTS DATA_ENGINEER;
DROP ROLE IF EXISTS EXECUTIVE;

SELECT 'Cleanup complete' AS status;